In [1]:
import pandas as pd
import numpy as np

from ripser import ripser
from sklearn.preprocessing import StandardScaler


# ==========================================
# 1. Load Data
# ==========================================

df = pd.read_csv(
    "data/market_data_labeled.csv",
    parse_dates=["Date"],
    index_col="Date"
)

print("Dataset Shape:", df.shape)


# ==========================================
# 2. Configuration
# ==========================================

WINDOW_SIZE = 30

tda_columns = [
    "SP500_Return",
    "NASDAQ_Return",
    "VIX_Change",
    "GOLD_Return",
    "OIL_Return"
]


# ==========================================
# 3. Persistence Feature Function
# ==========================================

def persistence_features(point_cloud):

    result = ripser(
        point_cloud,
        maxdim=1
    )

    diagrams = result["dgms"]

    features = {}

    # --------------------------------------
    # H0 Features
    # --------------------------------------

    h0 = diagrams[0]

    # Remove infinite persistence component
    h0 = h0[np.isfinite(h0[:, 1])]

    if len(h0) > 0:

        h0_persistence = (
            h0[:, 1] - h0[:, 0]
        )

        features["TDA_H0_Count"] = len(h0)

        features["TDA_H0_TotalPersistence"] = (
            np.sum(h0_persistence)
        )

        features["TDA_H0_MaxPersistence"] = (
            np.max(h0_persistence)
        )

        features["TDA_H0_MeanPersistence"] = (
            np.mean(h0_persistence)
        )

    else:

        features["TDA_H0_Count"] = 0
        features["TDA_H0_TotalPersistence"] = 0
        features["TDA_H0_MaxPersistence"] = 0
        features["TDA_H0_MeanPersistence"] = 0


    # --------------------------------------
    # H1 Features
    # --------------------------------------

    h1 = diagrams[1]

    if len(h1) > 0:

        h1_persistence = (
            h1[:, 1] - h1[:, 0]
        )

        features["TDA_H1_Count"] = len(h1)

        features["TDA_H1_TotalPersistence"] = (
            np.sum(h1_persistence)
        )

        features["TDA_H1_MaxPersistence"] = (
            np.max(h1_persistence)
        )

        features["TDA_H1_MeanPersistence"] = (
            np.mean(h1_persistence)
        )


        # ----------------------------------
        # Persistence Entropy
        # ----------------------------------

        total = np.sum(h1_persistence)

        if total > 0:

            probabilities = (
                h1_persistence / total
            )

            entropy = -np.sum(
                probabilities *
                np.log(
                    probabilities + 1e-12
                )
            )

        else:

            entropy = 0

        features["TDA_H1_Entropy"] = entropy

    else:

        features["TDA_H1_Count"] = 0
        features["TDA_H1_TotalPersistence"] = 0
        features["TDA_H1_MaxPersistence"] = 0
        features["TDA_H1_MeanPersistence"] = 0
        features["TDA_H1_Entropy"] = 0

    return features


# ==========================================
# 4. Sliding Window TDA
# ==========================================

tda_results = []

total_windows = (
    len(df) - WINDOW_SIZE + 1
)

print(
    "\nCalculating Persistent Homology..."
)

for end_idx in range(
    WINDOW_SIZE - 1,
    len(df)
):

    start_idx = (
        end_idx - WINDOW_SIZE + 1
    )

    window = df.iloc[
        start_idx:end_idx + 1
    ]

    point_cloud = (
        window[tda_columns]
        .replace(
            [np.inf, -np.inf],
            np.nan
        )
        .dropna()
        .values
    )

    if len(point_cloud) < WINDOW_SIZE:

        continue


    # --------------------------------------
    # Standardize only current/past window
    # --------------------------------------

    scaler = StandardScaler()

    point_cloud_scaled = (
        scaler.fit_transform(
            point_cloud
        )
    )


    # --------------------------------------
    # Persistent Homology
    # --------------------------------------

    features = persistence_features(
        point_cloud_scaled
    )

    features["Date"] = df.index[end_idx]

    tda_results.append(features)


    # Progress
    processed = (
        end_idx - WINDOW_SIZE + 2
    )

    if processed % 500 == 0:

        print(
            f"Processed "
            f"{processed}/"
            f"{total_windows}"
        )


# ==========================================
# 5. Create TDA DataFrame
# ==========================================

tda_df = pd.DataFrame(
    tda_results
)

tda_df.set_index(
    "Date",
    inplace=True
)

print(
    "\nTDA Dataset Shape:",
    tda_df.shape
)

print(
    "\nTDA Features:"
)

print(
    tda_df.columns.tolist()
)

print(
    "\nSample:"
)

print(
    tda_df.head()
)


# ==========================================
# 6. Validate
# ==========================================

print(
    "\nMissing Values:",
    tda_df.isnull().sum().sum()
)

print(
    "\nH1 Loop Statistics:"
)

print(
    tda_df[
        [
            "TDA_H1_Count",
            "TDA_H1_TotalPersistence",
            "TDA_H1_MaxPersistence",
            "TDA_H1_Entropy"
        ]
    ].describe()
)


# ==========================================
# 7. Merge with Main Dataset
# ==========================================

final_df = df.join(
    tda_df,
    how="inner"
)

print(
    "\nFinal Dataset Shape:",
    final_df.shape
)


# ==========================================
# 8. Save
# ==========================================

final_df.to_csv(
    "data/market_data_tda.csv"
)

print(
    "\nSaved: data/market_data_tda.csv"
)

Dataset Shape: (6452, 29)

Calculating Persistent Homology...
Processed 500/6423
Processed 1000/6423
Processed 1500/6423
Processed 2000/6423
Processed 2500/6423
Processed 3000/6423
Processed 3500/6423
Processed 4000/6423
Processed 4500/6423
Processed 5000/6423
Processed 5500/6423
Processed 6000/6423

TDA Dataset Shape: (6423, 9)

TDA Features:
['TDA_H0_Count', 'TDA_H0_TotalPersistence', 'TDA_H0_MaxPersistence', 'TDA_H0_MeanPersistence', 'TDA_H1_Count', 'TDA_H1_TotalPersistence', 'TDA_H1_MaxPersistence', 'TDA_H1_MeanPersistence', 'TDA_H1_Entropy']

Sample:
            TDA_H0_Count  TDA_H0_TotalPersistence  TDA_H0_MaxPersistence  \
Date                                                                       
2000-12-20            29                36.639742               3.407056   
2000-12-21            29                36.998494               3.375432   
2000-12-22            29                36.227031               3.375383   
2000-12-26            29                35.753402         